---
## 1. Data Definition and Preparation

### 1.1 Imports and Initial Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntSlider
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('superstore_dataset.csv', encoding='latin-1')

# Basic data exploration
print('Dataset Shape:', df.shape)
print('\nColumn Names:')
print(df.columns.tolist())

In [ ]:
# Detailed schema inspection
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Missing values audit
print('Missing values per column:')
print(df.isnull().sum())

### 1.2 Data Cleaning



In [ ]:
# Check and remove duplicates
print('Duplicate rows before cleaning:', df.duplicated().sum())
df = df.drop_duplicates()
print('Duplicate rows after cleaning:', df.duplicated().sum())

# Handle missing Postal Code
if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].fillna(0).astype(int)
    print('\nPostal Code nulls after fill:', df['Postal Code'].isnull().sum())

### 1.3 Data Type Fixes



In [ ]:
# Convert date columns
date_columns = ['Order Date', 'Ship Date']
for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

print('Data types after conversion:')
print(df[date_columns].dtypes)

### 1.4 Feature Engineering



In [ ]:
# Feature engineering
df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

print('New features - sample:')
print(df[['Sales', 'Profit', 'Profit Margin', 'Order Year', 'Order Month']].head(10))

---
## 2. Deep-Dive Exploratory Analysis (Matplotlib)

In [ ]:
# Prepare monthly aggregation
monthly_sales = df.groupby(['Order Month-Year', 'Category'])['Sales'].sum().reset_index()
monthly_sales['Date'] = monthly_sales['Order Month-Year'].dt.to_timestamp()

def plot_monthly_sales(category='All'):
    fig, ax = plt.subplots(figsize=(13, 6))

    if category == 'All':
        total_monthly = df.groupby('Order Month-Year')['Sales'].sum()
        ax.plot(total_monthly.index.to_timestamp(), total_monthly.values,
                marker='o', linewidth=2, markersize=4, color='steelblue')
        ax.set_title('Monthly Sales Trend - All Categories', fontsize=16, fontweight='bold')
    else:
        cat_data = monthly_sales[monthly_sales['Category'] == category]
        ax.plot(cat_data['Date'], cat_data['Sales'],
                marker='o', linewidth=2, markersize=4, color='darkorange')
        ax.set_title(f'Monthly Sales Trend - {category}', fontsize=16, fontweight='bold')

    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Sales (USD)', fontsize=12)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

categories = ['All'] + sorted(df['Category'].unique().tolist())
category_dropdown = Dropdown(options=categories, value='All', description='Category:')
interact(plot_monthly_sales, category=category_dropdown);

### 2.2 Geographic Sales Performance



In [ ]:
# Prepare state-level aggregation
state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=True)

def plot_top_states(top_n=10):
    fig, ax = plt.subplots(figsize=(13, max(6, top_n * 0.45)))

    top_states = state_sales.tail(top_n)
    colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(top_states)))

    ax.barh(range(len(top_states)), top_states.values, color=colors)
    ax.set_yticks(range(len(top_states)))
    ax.set_yticklabels(top_states.index, fontsize=10)
    ax.set_xlabel('Total Sales (USD)', fontsize=12)
    ax.set_ylabel('State', fontsize=12)
    ax.set_title(f'Top {top_n} States by Sales Performance', fontsize=16, fontweight='bold')

    max_val = top_states.values.max()
    for i, (state, value) in enumerate(top_states.items()):
        ax.text(value + max_val * 0.01, i, f'${value:,.0f}', va='center', fontsize=9)

    ax.grid(axis='x', alpha=0.3)
    fig.tight_layout()
    plt.show()

    print(f'Total states in dataset: {len(state_sales)}')
    print(f'Top {top_n} states combined revenue: ${top_states.sum():,.0f}')
    total_sales = df['Sales'].sum()
    print(f'Share of total revenue: {top_states.sum() / total_sales * 100:.1f}%')

top_n_slider = IntSlider(min=5, max=25, value=10, step=1, description='Top N States:')
interact(plot_top_states, top_n=top_n_slider);

---
## 3. Communicating Insights (Seaborn)

In [ ]:
# Top 10 most profitable products
product_profit = (
    df.groupby('Product Name')['Profit']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(13, 8))
sns.barplot(
    x=product_profit.values,
    y=product_profit.index,
    palette='viridis',
    orient='h',
    ax=ax
)

ax.set_title(
    'Top 10 Most Profitable Products\nExecutive Summary - Product Performance Analysis',
    fontsize=15, fontweight='bold', pad=20
)
ax.set_xlabel('Total Profit (USD)', fontsize=12, fontweight='bold')
ax.set_ylabel('Product Name', fontsize=12, fontweight='bold')

max_val = product_profit.values.max()
for i, (product, profit) in enumerate(product_profit.items()):
    ax.text(profit + max_val * 0.01, i, f'${profit:,.0f}',
            va='center', fontweight='bold', fontsize=10)

ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
plt.show()

print('Key Insights:')
print(f'  Most profitable product generates: ${product_profit.iloc[0]:,.0f}')
print(f'  Top 10 products combined profit:  ${product_profit.sum():,.0f}')
print(f'  Average profit per top product:   ${product_profit.mean():,.0f}')

### 3.2 Discount vs Profit Scatter Plot



In [ ]:
# Discount vs Profit analysis
fig, ax = plt.subplots(figsize=(14, 8))

sns.scatterplot(
    data=df, x='Discount', y='Profit',
    hue='Category', alpha=0.55, s=50, ax=ax
)

sns.regplot(
    data=df, x='Discount', y='Profit',
    scatter=False, color='red',
    line_kws={'linewidth': 2, 'linestyle': '--'},
    ax=ax
)

ax.axhline(y=0, color='black', linestyle='-', alpha=0.4, linewidth=1)
ax.text(0.52, 30, 'Break-even line', fontsize=10, alpha=0.6)

ax.set_title(
    'Discount Strategy Analysis: Impact on Profitability by Category',
    fontsize=15, fontweight='bold', pad=20
)
ax.set_xlabel('Discount Rate', fontsize=12, fontweight='bold')
ax.set_ylabel('Profit (USD)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(title='Product Category', bbox_to_anchor=(1.05, 1), loc='upper left')

fig.tight_layout()
plt.show()

# Analytical insights
high_discount = df[df['Discount'] > 0.20]
print('Discount Analysis Insights:')
print(f'  Transactions with >20% discount: {len(high_discount):,}')
print(f'  Average profit at >20% discount: ${high_discount["Profit"].mean():.2f}')
print(f'  Loss rate for >20% discounts:    {(high_discount["Profit"] < 0).mean()*100:.1f}%')

print('\nCategory-specific impact at >20% discount:')
for category in sorted(df['Category'].unique()):
    cat_hi = df[(df['Category'] == category) & (df['Discount'] > 0.20)]
    if len(cat_hi) > 0:
        print(f'  {category}: avg profit = ${cat_hi["Profit"].mean():.2f}  '
              f'(loss rate: {(cat_hi["Profit"] < 0).mean()*100:.1f}%)')

---
## 4. Methodology and Tooling Review



In [ ]:
import time

# Render speed comparison
start = time.time()
fig, ax = plt.subplots(figsize=(8, 5))
yearly = df.groupby('Order Year')['Sales'].sum()
ax.plot(yearly.index, yearly.values)
plt.close(fig)
matplotlib_time = time.time() - start

start = time.time()
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(
    data=df.groupby('Order Year')['Sales'].sum().reset_index(),
    x='Order Year', y='Sales', ax=ax
)
plt.close(fig)
seaborn_time = time.time() - start

print(f'Matplotlib basic line plot: {matplotlib_time:.4f}s')
print(f'Seaborn equivalent:         {seaborn_time:.4f}s')
print(f'Overhead ratio:             {seaborn_time/matplotlib_time:.1f}x')

---
## 5. Executive Summary and Strategic Recommendations

In [ ]:
# Recompute summary variables (safe to re-run in any order)
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
profit_margin = (total_profit / total_sales) * 100

state_sales_desc = df.groupby('State')['Sales'].sum().sort_values(ascending=False)
top_state        = state_sales_desc.index[0]
top_state_sales  = state_sales_desc.iloc[0]
top5_share       = state_sales_desc.head(5).sum() / total_sales * 100

top_category     = df.groupby('Category')['Sales'].sum().idxmax()
top_product      = df.groupby('Product Name')['Profit'].sum().idxmax()

high_disc_loss   = (df[df['Discount'] > 0.20]['Profit'] < 0).mean() * 100

print('=== EXECUTIVE SUMMARY - KEY FINDINGS ===')
print()
print('BUSINESS PERFORMANCE')
print(f'  Total Revenue:          ${total_sales:>12,.0f}')
print(f'  Total Profit:           ${total_profit:>12,.0f}')
print(f'  Overall Profit Margin:  {profit_margin:>11.1f}%')
print()
print('GEOGRAPHIC PERFORMANCE')
print(f'  Top performing state:   {top_state} (${top_state_sales:,.0f})')
print(f'  Top 5 states share:     {top5_share:.1f}% of total revenue')
print()
print('PRODUCT PERFORMANCE')
print(f'  Leading category:       {top_category}')
print(f'  Most profitable product:{top_product[:60]}')
print()
print('DISCOUNT STRATEGY RISK')
print(f'  >20% discount loss rate: {high_disc_loss:.1f}% of those transactions are unprofitable')
print(f'  Recommended cap:         20% discount to protect margin')

### Strategic Recommendations


---
## Advanced Section (Optional): Interactive Multi-Chart Dashboard

In [ ]:
def create_dashboard():
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(17, 12))
    fig.suptitle('US Superstore - Business Intelligence Dashboard', fontsize=18, fontweight='bold', y=1.01)

    # Chart 1: Monthly sales trend
    monthly_total = df.groupby('Order Month-Year')['Sales'].sum()
    ax1.plot(monthly_total.index.to_timestamp(), monthly_total.values,
             marker='o', markersize=3, linewidth=1.8, color='steelblue')
    ax1.set_title('Monthly Sales Trend', fontweight='bold')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Sales (USD)')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3)

    # Chart 2: Sales by category
    category_sales = df.groupby('Category')['Sales'].sum().sort_values()
    colors2 = ['#4c78a8', '#72b7b2', '#54a24b']
    ax2.bar(category_sales.index, category_sales.values, color=colors2)
    ax2.set_title('Sales by Category', fontweight='bold')
    ax2.set_ylabel('Sales (USD)')
    for i, (cat, val) in enumerate(category_sales.items()):
        ax2.text(i, val + category_sales.max() * 0.01, f'${val:,.0f}',
                 ha='center', fontsize=9, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)

    # Chart 3: Top 10 states
    top10 = state_sales.tail(10)
    bar_colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(top10)))
    ax3.barh(range(len(top10)), top10.values, color=bar_colors)
    ax3.set_yticks(range(len(top10)))
    ax3.set_yticklabels(top10.index, fontsize=9)
    ax3.set_title('Top 10 States by Sales', fontweight='bold')
    ax3.set_xlabel('Sales (USD)')
    ax3.grid(axis='x', alpha=0.3)

    # Chart 4: Discount vs Profit by category
    palette = {'Furniture': '#4c78a8', 'Office Supplies': '#72b7b2', 'Technology': '#f58518'}
    for category in df['Category'].unique():
        cat_data = df[df['Category'] == category]
        ax4.scatter(cat_data['Discount'], cat_data['Profit'],
                    label=category, alpha=0.5, s=20,
                    color=palette.get(category, 'grey'))
    ax4.axhline(y=0, color='black', linestyle='--', alpha=0.5, linewidth=1)
    ax4.set_title('Discount vs Profit by Category', fontweight='bold')
    ax4.set_xlabel('Discount Rate')
    ax4.set_ylabel('Profit (USD)')
    ax4.legend(fontsize=9)
    ax4.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

create_dashboard()

### Advanced: Outlier Annotation on Discount vs Profit

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))

sns.scatterplot(
    data=df, x='Discount', y='Profit',
    hue='Category', alpha=0.5, s=40, ax=ax
)

# Identify outliers
top3    = df.nlargest(3, 'Profit')
bottom3 = df.nsmallest(3, 'Profit')

# Annotate top 3 most profitable
for _, row in top3.iterrows():
    ax.annotate(
        f'Best: ${row["Profit"]:.0f}',
        xy=(row['Discount'], row['Profit']),
        xytext=(12, 8), textcoords='offset points',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.8),
        arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.2'),
        fontsize=9
    )

# Annotate bottom 3 least profitable
for _, row in bottom3.iterrows():
    ax.annotate(
        f'Worst: ${row["Profit"]:.0f}',
        xy=(row['Discount'], row['Profit']),
        xytext=(12, -15), textcoords='offset points',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='salmon', alpha=0.8),
        arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.2'),
        fontsize=9
    )

ax.axhline(y=0, color='black', linestyle='--', alpha=0.4, linewidth=1)
ax.set_title('Discount vs Profit - Outlier Identification', fontsize=15, fontweight='bold')
ax.set_xlabel('Discount Rate', fontsize=12)
ax.set_ylabel('Profit (USD)', fontsize=12)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

### Advanced: Plotly Express Comparison

In [ ]:
try:
    import plotly.express as px

    fig_px = px.scatter(
        df, x='Discount', y='Profit',
        color='Category',
        hover_data=['Product Name', 'Sales'],
        trendline='ols',
        title='Interactive Discount vs Profit Analysis (Plotly Express)'
    )
    fig_px.show()

except ImportError:
    print('Plotly is not installed. Run: pip install plotly statsmodels')

print()
print('PLOTLY vs MATPLOTLIB - COMPARISON')
print()
print('Plotly advantages:')
print('  - Built-in zoom, pan, and hover tooltips without extra code')
print('  - Easily shareable as standalone HTML files')
print('  - Trendlines and statistical layers available as one-liners')
print('  - Responsive layout adapts automatically to screen size')
print()
print('Matplotlib + ipywidgets advantages:')
print('  - Granular control over every visual element')
print('  - Seamless widget integration for multi-parameter interactivity')
print('  - Smaller rendered output size (SVG/PNG vs interactive JS bundle)')
print('  - Broader familiarity among data science teams')